# UniVTG CSV Preview Pipeline V2

이 노트북은 **설정과 실행만 담당**합니다. CSV 검증, 캐시, UniVTG 추론, 점수 결합, 장면 선택, 렌더링은 `univtg_preview` 패키지로 분리되어 있습니다.

입력 파일은 기본적으로 두 개입니다.

1. `query_dataset.csv`: `fixed_query_1~6`, `indiv_query_1~6`
2. `intro_timing_dataset.csv`: 인트로 종료, 브랜딩 종료, 영상 종료 시간

현재 `query_dataset.csv` 안에 fixed Query가 있으므로 `category_query_dataset.csv`는 사용하지 않습니다.

팀원이 Query를 두 파일로 나누어 전달하면 `QUERY_CSV`에는 영상별 `indiv_query_1~6` 파일을 지정하고, `CATEGORY_QUERY_CSV`에는 카테고리별 `fixed_query_1~6` 파일을 지정합니다.

## 0. Google Drive 경로 설정

`univtg_preview_v2.zip`을 Drive의 `code` 폴더에 올리고, Query CSV와 timing CSV를 `datasets` 폴더에 둡니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/'
    '0_ASAC_11기_DL_3조'
)

PACKAGE_ZIP = PROJECT_ROOT / 'code' / 'univtg_preview_v2.zip'
PACKAGE_DIR = Path('/content/univtg_preview_v2')

QUERY_CSV = PROJECT_ROOT / 'datasets' / 'query_dataset.csv'
TIMING_CSV = PROJECT_ROOT / 'datasets' / 'intro_timing_dataset.csv'
PROFILE_YAML = PACKAGE_DIR / 'configs' / 'category_profiles.yaml'

# 현재 Query CSV 안에 fixed_query_1~6이 있으므로 None입니다.
CATEGORY_QUERY_CSV = None

WORKSPACE_ROOT = Path('/content/univtg_workspace')
DRIVE_EXPORT_ROOT = PROJECT_ROOT / 'outputs' / 'univtg_preview_v2'

assert PACKAGE_ZIP.exists(), f'패키지 ZIP을 찾지 못했습니다: {PACKAGE_ZIP}'
assert QUERY_CSV.exists(), f'Query CSV를 찾지 못했습니다: {QUERY_CSV}'
assert TIMING_CSV.exists(), f'Timing CSV를 찾지 못했습니다: {TIMING_CSV}'

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
shutil.unpack_archive(str(PACKAGE_ZIP), '/content')

if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))

print('Package:', PACKAGE_DIR)
print('Query CSV:', QUERY_CSV)
print('Timing CSV:', TIMING_CSV)
print('Export:', DRIVE_EXPORT_ROOT)

## 1. UniVTG 환경 준비

공식 저장소는 고정 commit으로 checkout하고, 공식 checkpoint와 CLIP/UniVTG 모델은 런타임당 한 번만 불러옵니다.

In [ ]:
from univtg_preview.colab_setup import (
    ensure_checkpoint,
    ensure_repository,
    install_dependencies,
    load_models,
)

install_dependencies()
REPO_DIR = ensure_repository()
CHECKPOINT, OPT_JSON = ensure_checkpoint(REPO_DIR)
MODEL_BUNDLE = load_models(
    repo_dir=REPO_DIR,
    checkpoint=CHECKPOINT,
)

print('Repository:', REPO_DIR)
print('Checkpoint:', CHECKPOINT)
print('Options:', OPT_JSON)

## 2. 두 CSV 로드 및 검증

이 셀에서 다음을 한 번에 검사합니다.

- `video_id` 중복과 NFC/NFD 한글 차이
- Query 6개 누락
- category fixed Query 일관성
- `intro_start <= intro_end <= body_start < video_end`
- 문자열 `False`/`True` 처리
- Query CSV와 timing CSV의 `video_id` join

In [ ]:
from IPython.display import display

from univtg_preview import (
    DatasetRepository,
    ExperimentFactory,
    load_category_profiles,
)

repository = DatasetRepository.from_csv(
    query_csv=QUERY_CSV,
    timing_csv=TIMING_CSV,
    category_query_csv=CATEGORY_QUERY_CSV,
)
profiles = load_category_profiles(PROFILE_YAML)
factory = ExperimentFactory(
    repository=repository,
    profiles=profiles,
)

display(repository.summary())
print('Video count:', len(repository.video_ids()))
print('Categories:', sorted(profiles))

## 3. Pipeline 객체 생성

- 영상과 `vid.npz`: Query 종류와 무관한 공통 cache
- `txt.npz`: Query 문자열 hash 기반 cache
- 결과: fixed/individual별 독립 run 폴더
- 최종 결과: Google Drive `outputs/univtg_preview_v2`로 복사

In [ ]:
from univtg_preview import (
    PreviewPipeline,
    UniVTGRuntime,
    Workspace,
)

runtime = UniVTGRuntime(
    clip_model=MODEL_BUNDLE.clip_model,
    univtg_model=MODEL_BUNDLE.univtg_model,
    device=MODEL_BUNDLE.device,
    vid2clip=MODEL_BUNDLE.vid2clip,
    txt2clip=MODEL_BUNDLE.txt2clip,
    l2_normalize_np_array=MODEL_BUNDLE.l2_normalize_np_array,
    clip_model_name=MODEL_BUNDLE.clip_model_name,
)

workspace = Workspace(
    root=WORKSPACE_ROOT,
    export_root=DRIVE_EXPORT_ROOT,
)

pipeline = PreviewPipeline(
    runtime=runtime,
    workspace=workspace,
)

print('Pipeline ready')

## 4. 실행 전 spec 확인

수동 `GAME_CONFIG = {...}` 블록을 만들지 않습니다. `video_id`와 `query_mode`만 지정하면 CSV와 category profile에서 설정이 생성됩니다.

In [ ]:
import pandas as pd

GAME_VIDEO_ID = '혜안_네모네모'

game_fixed_spec = factory.build(
    GAME_VIDEO_ID,
    'fixed',
)

display(pd.DataFrame({
    'item': [
        'video_id',
        'category',
        'query_mode',
        'query_count',
        'intro_range_sec',
        'body_start_sec',
        'requested_video_end_sec',
        'analysis_clip_len_sec',
        'scene_duration_sec',
        'num_scenes',
        'expected_preview_sec',
    ],
    'value': [
        game_fixed_spec.video_id,
        game_fixed_spec.category,
        game_fixed_spec.query_mode.value,
        len(game_fixed_spec.queries),
        f'{game_fixed_spec.timing.intro_start_sec}~{game_fixed_spec.timing.intro_end_sec}',
        game_fixed_spec.timing.body_start_sec,
        game_fixed_spec.timing.requested_video_end_sec,
        game_fixed_spec.profile.analysis_clip_len_sec,
        game_fixed_spec.profile.scene_duration_sec,
        game_fixed_spec.profile.num_scenes,
        game_fixed_spec.profile.expected_preview_duration_sec,
    ],
}))

print('
Queries')
for index, query in enumerate(game_fixed_spec.queries, start=1):
    print(f'{index}. {query}')

## 5. 게임 고정 Query 실행

In [ ]:
from IPython.display import Video, display

game_fixed_result = pipeline.run(game_fixed_spec)
print(game_fixed_result.to_dict())

display(Video(
    str(game_fixed_result.paths.preview_video),
    embed=True,
    width=720,
))
display(pipeline.load_selected_preview(game_fixed_result))

## 6. 같은 게임 영상의 LLM 개별 Query 실행

본문 MP4와 `vid.npz`는 고정 Query 실험에서 만든 cache를 그대로 사용합니다. Query text feature와 추론·후처리만 새로 수행합니다.

In [ ]:
game_individual_spec = factory.build(
    GAME_VIDEO_ID,
    'individual',
)

game_individual_result = pipeline.run(game_individual_spec)
print(game_individual_result.to_dict())

display(Video(
    str(game_individual_result.paths.preview_video),
    embed=True,
    width=720,
))
display(pipeline.load_selected_preview(game_individual_result))

## 7. fixed와 individual 선택 결과 비교

In [ ]:
fixed_selected = pipeline.load_selected_preview(game_fixed_result).assign(
    query_mode='fixed'
)
individual_selected = pipeline.load_selected_preview(game_individual_result).assign(
    query_mode='individual'
)

comparison = pd.concat(
    [fixed_selected, individual_selected],
    ignore_index=True,
)

display(comparison[[
    'query_mode',
    'preview_order',
    'score_rank',
    'segment_start_full_sec',
    'segment_end_full_sec',
    'combined_score',
    'best_query',
]])

## 8. 드라마 실행

In [ ]:
DRAMA_VIDEO_ID = '드라마_소용없어거짓말'

drama_individual_spec = factory.build(
    DRAMA_VIDEO_ID,
    'individual',
)

drama_individual_result = pipeline.run(drama_individual_spec)
print(drama_individual_result.to_dict())

display(Video(
    str(drama_individual_result.paths.preview_video),
    embed=True,
    width=720,
))

## 9. 파일럿 배치 실행

처음에는 3개만 실행합니다. 각 영상이 끝날 때마다 status CSV를 저장하므로 런타임이 중단되어도 `resume=True`로 성공한 조합을 건너뜁니다.

In [ ]:
from univtg_preview import BatchRunner

batch_runner = BatchRunner(
    factory=factory,
    pipeline=pipeline,
    status_csv=DRIVE_EXPORT_ROOT / 'batch_status.csv',
)

pilot_status = batch_runner.run(
    query_modes=['fixed', 'individual'],
    categories=['game'],
    limit=3,
    resume=True,
    export=True,
)

display(pilot_status.tail(20))

## 10. 전체 Train 실행 예시

`query_dataset.csv`에 `split` 컬럼이 있을 때 아래 셀을 사용합니다. 먼저 파일럿 결과를 확인한 뒤 실행합니다.

In [ ]:
# train_status = batch_runner.run(
#     query_modes=['fixed', 'individual'],
#     splits=['train'],
#     limit=None,
#     resume=True,
#     export=True,
# )
# display(train_status.tail(20))

## 11. 설정을 바꾸는 위치

영상별 실행 셀을 수정하지 않고 `configs/category_profiles.yaml`만 변경합니다.

- 게임 2초 × 10개: `game`
- 드라마 10초 × 5개: `drama`
- 팟캐스트 15초 × 4개: `podcast`
- 영화 리뷰 15초 × 4개: `movie_review`

특정 영상만 임시로 바꾸려면 다음처럼 profile override를 사용합니다.

```python
spec = factory.build(
    'VIDEO_ID',
    'individual',
    profile_overrides={
        'num_scenes': 6,
        'min_scene_gap_sec': 3.0,
    },
)
```
